## Batch mode

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService, Batch, SamplerV2 as Sampler, EstimatorV2 as Estimator

service = QiskitRuntimeService()

In [8]:
backend = service.backend('ibm_basquecountry')
batch = Batch(backend=backend)
sampler = Sampler(mode=batch)
estimator = Estimator(mode=batch)
batch.close() # or we can also do the following

In [9]:
with Batch(backend=backend):
    estimator = Estimator()
    sampler = Sampler()

In [10]:
# We can also define a duration on the batch
with Batch(backend=backend, max_time='25s'):
    estimator = Estimator()

In [13]:
# Let's try run a simple code
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.cx(0,1)
qc1.measure_all()

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circ1 = pm.run(qc1)

qc2 = QuantumCircuit(2)
qc2.x(0)
qc2.h(0)
qc2.cx(0,1)
qc2.measure_all()

isa_circ2 = pm.run(qc2)

with Batch(backend=backend):
    sampler = Sampler()
    job1 = sampler.run([isa_circ1])
    job2 = sampler.run([isa_circ2])
result1 = job1.result()
result2 = job2.result()

print(result1[0].data.meas.get_counts())
print(result2[0].data.meas.get_counts())


{'00': 2026, '11': 1974, '01': 44, '10': 52}
{'11': 2006, '00': 1973, '10': 62, '01': 55}


## Session mode

In [14]:
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
service = QiskitRuntimeService()
backend = service.backend('ibm_basquecountry')
session = Session(backend=backend)
sampler = Sampler(mode=session)
job1 = sampler.run([isa_circ1])
job2 = sampler.run([isa_circ2])
session.close()
result1 = job1.result()
result2 = job2.result()

In [15]:
print(result1[0].data.meas.get_counts())
print(result2[0].data.meas.get_counts())

{'11': 1933, '00': 2049, '01': 63, '10': 51}
{'00': 2004, '11': 1985, '01': 53, '10': 54}


In [ ]:
# Same as before with the other option
with Session(backend=backend):
    sampler = Sampler()
    job1 = sampler.run([isa_circ1])
    job2 = sampler.run([isa_circ2])
result1 = job1.result()
result2 = job2.result()
print(result1[0].data.meas.get_counts())
print(result2[0].data.meas.get_counts())
